### Import all the required packages for developing the models

In [2]:
import pandas as pd
import numpy as np
import faiss
import requests

In [ ]:
dataset = pd.read_csv('dataset/books.csv')

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [4]:
def textual_representation(row):
    """Iterates through all the rows in the dataset and only extracts the necessary information and store it in a variable
    as String named textual_representation
    
    input: row -> parameter, datatype ->  DataFrame Object
    output: return textual_representation -> String 
    """
    textual_representation = f"""
    Title: {row['title']}
    Authors: {row['authors']}
    Description: {row['description']}
    Categories: {row['categories']}
    Publishing Year: {row['published_year']}
    Average Rating: {row['average_rating']}
    Number of pages: {row['num_pages']}"""

    return textual_representation

#### Apply textual representation to all rows in the dataframe and add that as a column in the dataframe

In [ ]:
dataset['textual_representation'] = dataset.apply(textual_representation, axis=1)

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,textual_representation


#### Take all the textual representation and make an embedding of it using an LLM and add it to the faiss(FaceBook AI similarity search) vector store

In [6]:
dimensions = 4096

index = faiss.IndexFlatL2(dimensions)

X = np.zeros((len(dataset['textual_representation']), dimensions), dtype='float32')

In [12]:
for i, representation in enumerate(dataset['textual_representation']):
    if i > 500:
        break;

    response = requests.post('http://localhost:11434/api/embeddings',
                        json = {
                            'model': 'llama2',
                            'prompt': representation
                        })

    embeddings = response.json()['embedding']

    X[i] = np.array(embeddings)

index.add(X)

In [13]:
faiss.write_index(index, 'index')

In [ ]:
input_book = dataset.iloc[1000]


    Title: Heat
    Authors: Mike Lupica
    Description: Pitching prodigy Michael Arroyo is on the run from social services after being banned from playing Little League baseball because rival coaches doubt he is only twelve years old and he has no parents to offer them proof. Reprint.
    Categories: Juvenile Fiction
    Publishing Year: 2007.0
    Average Rating: 3.98
    Number of pages: 220.0


In [1]:
def model_result(query):
    user_input_response = requests.post('http://localhost:11434/api/embeddings',
                        json = {
                            'model': 'llama2',
                            'prompt': query['textual_representation']
                        })
    user_input_embedding = np.array([user_input_response.json()['embedding']], dtype='float32')

    D, I = index.search(user_input_embedding, 5)

    best_matches = np.array(dataset['textual_representation'])[I.flatten()]

    return best_matches